# 로컬 트레이싱과 지연

Python 3.12 새 커널에서 RAG 단계별 로컬 실행 추적 준비부터 확인합니다.

## Python 3.12 패키지 버전 고정

In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## RAG 단계별 로컬 실행 추적 준비

In [2]:
# 한 대화의 재작성·검색·리랭킹·생성 구간을 밀리초 단위로 기록합니다.
trace = {
 'trace_id': 'local-trace-001', 'conversation_id': 'conv-001',
 'started_ms': 0, 'first_token_ms': 146, 'ended_ms': 301,
 'children': [
  {'name': 'rewrite', 'start': 0, 'end': 12},
  {'name': 'retrieve', 'start': 12, 'end': 43},
  {'name': 'rerank', 'start': 43, 'end': 61},
  {'name': 'generate', 'start': 61, 'end': 301},
 ],
}

## 첫 토큰 도착 시간과 전체 지연 계산

In [3]:
# 생성 시작 시각을 기준으로 TTFT를, 요청 시작을 기준으로 total을 계산합니다.
generate = next(x for x in trace['children'] if x['name'] == 'generate')
ttft = trace['first_token_ms'] - generate['start']
total = trace['ended_ms'] - trace['started_ms']
print('TTFT(ms):', ttft)
print('total(ms):', total)
for run in trace['children']:
    print(run['name'], run['end'] - run['start'])

TTFT(ms): 85
total(ms): 301
rewrite 12
retrieve 31
rerank 18
generate 240


## 캐시 읽기·생성 사용량 분리

In [4]:
# 입출력 토큰과 캐시 read·creation 토큰을 서로 다른 필드로 보존합니다.
usage = {
 'input_tokens': 1200, 'output_tokens': 180,
 'cache_read_tokens': 420, 'cache_creation_tokens': 80,
}
trace['usage'] = usage
print(usage)
assert ttft == 85 and total == 301
print('cloud 업로드: 미실행(LangSmith API 키 없음)')

{'input_tokens': 1200, 'output_tokens': 180, 'cache_read_tokens': 420, 'cache_creation_tokens': 80}
cloud 업로드: 미실행(LangSmith API 키 없음)
